## 1. Dataset Enrichment (with Token Metadata & Price Reference Data)

In order to model our transfer dataset properly, we need to enrich the dataset with the token metadata and price changes at the time of transfer. Normally, we have to ingest these token and price datasets separately because in our raw blockchain data, information like the price of a token is not associated when we extract the dataset. At this point, we perform dataset enrichment to merge these datasets together.

To achieve this, we use reference tables (often loaded via CSV files in tools like dbt) that contain the metadata and prices of the tokens involved. The purpose of these reference datasets is to complement the transactional data and act as a reliable source of context for valuation.



## 2. Setting Up reference CSVs (Seeds)

Create two CSV files inside your `seeds/` directory:

### A. `token_metadata.csv`
Contains static token profiles to translate raw contract addresses to names and decimal sizes:
```bash
token_address,symbol,decimals,token_name
0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,6,USD Coin
0xdac17f958d2ee523a2206206994597c13d831ec7,USDT,6,Tether
0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,WETH,18,Wrapped Ether
0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,8,Wrapped Bitcoin
```

### B. `token_prices.csv`
Contains historical hourly price data matching the block time ranges:
```bash
price_date,token_address,price_usd
2026-07-12,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,1.00
2026-07-12,0xdac17f958d2ee523a2206206994597c13d831ec7,1.00
2026-07-12,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,3500.00
2026-07-12,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,95000.00
```

Run:
```bash 
uv run dbt seed
``` 
dbt populates these tables inside your warehouse.

After a succesfull run , check your warehouse for this new added tables.

## 3. Designing a Three-Tier Modeling Pipeline

We will implement our models across three layers:

```
Raw Event logs  ──>  Staging (Casting/Renaming)  ──>  Intermediate (Joins/Math)  ──>  Marts (Dashboard Ready)
```

## 4. Writing the Transformation Models

### Tier 1: Staging Layer (`models/staging/stg_transfers.sql`) | find code [here](https://github.com/Analytic-Sages/data-engineering-course/blob/main/Module-03/files/stg_transfers.sql)
Casts and sanitizes raw log columns:

```bash
{{
    config(
        materialized = 'table',
        schema = 'staging',
        alias = 'staged_transfers',
        
    )
}}


with raw_transfers as (
    select * from {{ source('envio', 'Transfer') }}
)

select
    id as transfer_id,
    to_timestamp("blockTimestamp") as block_time,
    lower("from") as sender_address,
    lower("to") as recipient_address,
    value as raw_value,
    "blockNumber" as block_number,
    txhash,
    lower("tokenAddress") as token_address
from raw_transfers
where value > 0

```

### Tier 2: Intermediate Layer (`models/intermediate/int_transfers_normalized.sql`) | find code [here](https://github.com/Analytic-Sages/data-engineering-course/blob/main/Module-03/files/int_transfers.sql)
Joins raw events with decimal seeds to compute standard asset values:

```bash

{{
    config(
        materialized = 'table',
        alias='transfers',
        schema='intermediate'
        )
    }}

with staging_transfers as (
    select * from {{ref('stg_transfers')}}
),
meta as (
    select 
        lower(token_address) as token_address,
        symbol,
        decimals,
        token_name
    from {{ref('token_metadata')}}
)


select
    t.txhash,
    block_time,
    t.token_address,
    m.symbol,
    t.sender_address,
    t.recipient_address,
    t.raw_value,
    (t.raw_value / power(10, m.decimals)) as normalized_amount
from staging_transfers t
join meta m on t.token_address = m.token_address

```

### Tier 3: Marts Layer (`models/marts/fct_transfers_usd.sql`) | find code [here](https://github.com/Analytic-Sages/data-engineering-course/blob/main/Module-03/files/fct_transfers_usd.sql)
Joins normalized quantities with market price seeds to calculate transaction value in USD:

```bash
{{
    config(
        materialized='table',
        schema='fact',
        alias='transfers'
    )
}}


with normalized as (
    select * from {{ ref('int_transfers') }}
),
prices as (
    select * from {{ ref('token_prices') }}
)

select
    n.txhash,
    n.block_time,
    n.symbol,
    n.sender_address,
    n.recipient_address,
    n.token_address,
    n.normalized_amount,
    coalesce(p.price_usd, 0) as token_price,
    (n.normalized_amount * coalesce(p.price_usd, 0)) as usd_value
from normalized n
left join prices p 
    on n.token_address = p.token_address 
    and n.block_time = p.price_date
```

## 5. Compiling and Verifying the Pipeline

To compile and build your tables in the database, run:
```bash
uv run dbt run
```

Because of the `ref()` functions used inside our SQL files, dbt automatically resolves dependencies and builds the tables in the correct order: staging first, then intermediate, and finally marts.

---

## 6. Testing Our Models

To guarantee data quality and integrity, we must add data tests to validate our tables. We can configure these tests directly in our YAML files.

### A. Add Staging Tests
Add these test configurations to `models/staging/schema.yml` to verify that transaction hashes and token addresses are never null:

```yaml
version: 2

models:
  - name: stg_transfers
    description: "Cleaned token transfers table"
    columns:
      - name: txhash
        description: "The transaction hash"
        tests:
          - not_null
      - name: token_address
        description: "The token address"
        tests:
          - not_null
```

### B. Add Marts Tests
Add these test configurations to `models/mart/schema.yml` to make sure our final mart table contains no null identifiers, and that token prices are never negative:

```yaml
version: 2

models:
  - name: fact_transfers
    description: "Enriched token transfers mart table"
    columns:
      - name: txhash
        description: "The transaction hash"
        tests:
          - not_null
      - name: token_address
        description: "The token address"
        tests:
          - not_null
      - name: token_price
        description: "USD price of the token"
        tests:
          - is_non_negative
```

---

## 7. Verifying Our Data Quality

To execute all configured tests and verify your data, run:
```bash
uv run dbt test
```
*Note: If any tests fail, inspect the source data and staging rules to locate the source of the anomaly.*

### The Combined Build Command
Running `dbt run` and `dbt test` separately is common, but dbt provides a unified command that runs both operations sequentially in a single step:

```bash
uv run dbt build
```

The `dbt build` command runs all seeds, staging models, intermediate models, marts, and their corresponding data tests. If any step (or test) fails, downstream models are automatically skipped to prevent bad data from polluting your dashboard-ready tables.